<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 1:  Modelo CBOW (WordEmbeddings)</h1>
    <h2>Word Embeddings</h2>
</div>

En esta archivo de jupyter, obtendremos un modelo de CBOW desde cero para generar los wordembeddings. Saber cómo entrenar estos modelos nos brindará una mejor comprensión de los vectores palabras, que son los componentes básicos de muchas aplicaciones en el procesamiento del lenguaje natural.

<a name='1'></a>
# Continuous bag of words (CBOW)

Observemos la siguiente sentencia: 
>**'I am happy because I am learning'**. 

- Como lo vimos en la clase pasada, en el modelado de bolsa continua de palabras (CBOW), intentamos predecir la palabra central dadas algunas palabras de contexto (las palabras alrededor de la palabra central).

- Por ejemplo, si tuviera que elegir un contexto de tamaño medio, digamos $C = 2$, entonces intentaría predecir la palabra **happy** dado el contexto que incluye 2 palabras antes y 2 palabras después de la palabra central. :

> $C$ palabras antes: [I, am] 

> $C$ palabras después: [because, I] 

- en otras palabras el vector contexto, y el vector target quedarían como:

$$context = [I,am, because, I]$$
$$target = happy$$

Una estructura simple del modelo se ve como sigue:

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Figures/word2.png' alt="alternate text" width="width" height="height" style="width:600px;height:250px;" /></div>

donde $\bar x$ es el promedio de todos los vectores one-hot de las palabras de contexto.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='Figures/mean_vec2.png' alt="alternate text" width="width" height="height" style="width:600px;height:250px;" />  </div>

Una vez que ya tenemos los vectores contextos, podemos usar $\bar x$  como la entrada al modelo 

La arquitectura a implementar es la siguiente:

\begin{align}
 h &= W_1 \  X + b_1  \tag{1} \\
 a &= ReLU(h)  \tag{2} \\
 z &= W_2 \  a + b_2   \tag{3} \\
 \hat y &= softmax(z)   \tag{4} \\
\end{align}

In [ ]:
# Import Python libraries and helper functions (in utils2) 
import nltk
from nltk.tokenize import word_tokenize
import numpy as np
from collections import Counter
from utils2 import sigmoid, get_batches, compute_pca, get_dict
from NLP_TextCleaner import TextCleaningPipeline

In [ ]:
# Download sentence tokenizer
nltk.data.path.append('.')

In [ ]:
# Load, tokenize and process the data
import re                                                           # Cargando los datos del archivo de texto
with open('Data/shakespeare.txt') as f:
    data = f.read()

In [ ]:
data = re.sub(r'[,!?;-]', '.',data)                                 #simbolos de punctuacion es remplazado por .
tokens_data = nltk.word_tokenize(data)                                     #  Tokenizacion usando nltk 
tokens_data = [ch.lower() for ch in tokens_data if ch.isalpha() or ch == '.']     #  Normalización de los tokens
print(f"Número total de tokens:{len(tokens_data)} \n Ejemplo de un token en posición 15: {tokens_data[:15]}")         #  print data sample

In [ ]:
# Calculamos la distribución de frecuenciaas de palabras en el corpus
fdist = nltk.FreqDist(word for word in tokens_data)
vocabulario = set(tokens_data)
n_vocabulario = len(fdist)
print("Tamaño del vocabulario: ",n_vocabulario)
print("Los tokens más frecuentes: ",fdist.most_common(20) ) 

#### Mapear las palabras a sus indices y de indices a palabras


In [ ]:
# obtener los diccionarios palabras vs indices, indices vs palabras
word2Ind, Ind2word = get_dict(tokens_data)   #get_dic función en librería utils2


In [ ]:
Ind2word

In [ ]:
# example of word to index mapping
print("Indice de la palabra 'king' :  ",word2Ind['king'] )
print("Palabra cuyo indice es 2743:  ",Ind2word[2743] )

<a name='2'></a>
# Entrenando el Modelo

###  Inicialización del modelo

Ahora inicializaremos dos matrices y dos vectores.
- La primera matriz ($W_1$) es de dimensión $N \times V$, donde $V$ es el número de palabras en tu vocabulario y $N$ es la dimensión de tu vector de palabras.
- La segunda matriz ($W_2$) es de dimensión $V \times N$.
- El vector $b_1$ tiene dimensiones $N\times 1$
- El vector $b_2$ tiene dimensiones $V\times 1$.
- $b_1$ y $b_2$ son los vectores bias.

La estructura general del modelo se verá como en la Figura 1, pero en esta etapa solo estamos inicializando los parámetros.


In [ ]:
def initialize_model(N,V, random_seed=1):
    np.random.seed(random_seed)
    # W1 con shape (N,V)
    W1 = np.random.rand(N,V)
    # W2 con shape (V,N)
    W2 = np.random.rand(V,N)
    # b1 con shape (N,1)
    b1 = np.random.rand(N,1)
    # b2 con shape (V,1)
    b2 = np.random.rand(V,1)
    
    return W1, W2, b1, b2

In [ ]:
# Probamos la inicialización
tmp_N = 4
tmp_V = 10
tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(tmp_N,tmp_V)
print(f"tmp_W1.shape: {tmp_W1.shape}")
print(f"tmp_W2.shape: {tmp_W2.shape}")
print(f"tmp_b1.shape: {tmp_b1.shape}")
print(f"tmp_b2.shape: {tmp_b2.shape}")

<a name='2.1'></a>
### 2.1 Softmax
Antes de que podamos comenzar a entrenar el modelo, debemos implementar la función softmax:  

<br>
$$ \text{softmax}(z_i) = \frac{e^{z_i} }{\sum_{i=0}^{V-1} e^{z_i} }  $$

- $V$ es el número de palabras en el vocabulario (que también es el número de filas de $z$).
- $i$ va de 0 a |V| - 1.


**Implementando la función softmax.**

- Supongamos que la entrada $z$ a `softmax` es una matriz 2D
- Cada ejemplo de entrenamiento está representado por una columna de forma (V, 1) en esta matriz 2D.
- Puede haber más de una columna en la matriz 2D, porque puede incluir un lote de ejemplos para aumentar la eficiencia. Llamemos al tamaño del lote $m$ en minúsculas, por lo que la matriz $z$ tiene la forma (V, m)
- Al tomar la suma de $i=1 \cdots V-1$, toma la suma de cada columna (cada ejemplo) por separado.


In [ ]:
def softmax(z):
    # Calcular yhat (softmax)
    e_z = np.exp(z)
    yhat = e_z/np.sum(e_z,axis=0)    
    return yhat

In [ ]:
# Probando la función
tmp = np.array([[1,2,3],
                [1,1,1]
               ])
tmp_sm = softmax(tmp)
display(tmp_sm)

<a name='2.2'></a>
### Forward propagation

Implementemos la propagación directa $z$ según las ecuaciones. <br>

\begin{align}
 h &= W_1 \  X + b_1   \\
 a &= ReLU(h)   \\
 z &= W_2 \  a + b_2   \\
\end{align}

Para ello utilizaremos como función de activación la ReLU dada por:

$$f(h)=\max (0,h) \tag{6}$$

In [ ]:
def forward_prop(x, W1, W2, b1, b2):
    # Calculando h
    h = np.dot(W1,x)+b1
    
    # Aplicamos la función de activaación relu
    h = np.maximum(0,h)
    
    # Caalculando z
    z = np.dot(W2,h)+b2

    return z, h

In [ ]:
# Probando la función de forward_prop

# Creamos algunas entradas para probar la función

tmp_N = 2    #Numero de neuronas en capa oculta
tmp_V = 3    # tamaño del vocabulario

tmp_x = np.array([[0,1,0]]).T   # ejemplo de la representación de alguna palabra en el vocabulario

#Inicializamos la red
tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(N=tmp_N,V=tmp_V, random_seed=1)

print(f"x tiene un shape {tmp_x.shape}")
print(f"N es {tmp_N} y el tamaño del vocabulario V es {tmp_V}")

# llamando la función forward propagation
tmp_z, tmp_h = forward_prop(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)

#Veamos la salida
print(f"z tiene shape {tmp_z.shape}")
print(f"z contiene los valores:\n{tmp_z}")
print()
print(f"h tiene shape {tmp_h.shape}")
print("h tiene valores:")
print(tmp_h)


## Función de Costo


$$ J = -\frac{1}{n}\sum\limits_{k=1}^{V} y_k \log{\hat{y}_k} + (1-y_k)\log({1-{\hat{y}_k}})$$

In [ ]:
# Función de costo de entropía para clasificación
def compute_cost(y, yhat, batch_size):
    # cost function 
    logprobs = np.multiply(np.log(yhat),y) + np.multiply(np.log(1 - yhat), 1 - y)
    cost = - 1/batch_size * np.sum(logprobs)
    cost = np.squeeze(cost)
    return cost

In [ ]:
# Probando las funciones nuevamente
tmp_C = 2    
tmp_N = 50
tmp_batch_size = 4
tmp_word2Ind, tmp_Ind2word = get_dict(tokens_data)
tmp_V = len(word2Ind)

tmp_x, tmp_y = next(get_batches(tokens_data, tmp_word2Ind, tmp_V, tmp_C, tmp_batch_size))
        
print(f"tmp_x.shape {tmp_x.shape}")
print(f"tmp_y.shape {tmp_y.shape}")

tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(tmp_N,tmp_V)

print(f"tmp_W1.shape {tmp_W1.shape}")
print(f"tmp_W2.shape {tmp_W2.shape}")
print(f"tmp_b1.shape {tmp_b1.shape}")
print(f"tmp_b2.shape {tmp_b2.shape}")

tmp_z, tmp_h = forward_prop(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)
print(f"tmp_z.shape: {tmp_z.shape}")
print(f"tmp_h.shape: {tmp_h.shape}")

tmp_yhat = softmax(tmp_z)
print(f"tmp_yhat.shape: {tmp_yhat.shape}")

tmp_cost = compute_cost(tmp_y, tmp_yhat, tmp_batch_size)
print("Llamamos a la función de costo (compute_cost)")
print(f"tmp_cost {tmp_cost:.4f}")


## Training the Model - Backpropagation

Ahora entrenaremos el modelo CBOW <br>
Creamos una función para la propagación hacia adelante. Ahora implementaremos una función que calcula los gradientes para propagar los errores hacia atrás.

In [ ]:
def relu(z):
    result = z.copy()
    result[result<0]=0
    return result

In [ ]:
l1=np.array([[0.52727857,  0.52727857,  0.52727857,  0.52727857],
 [-0.1259346,  -0.1259346,  -0.1259346 , -0.1259346 ],
 [ 0.39739328,  0.39739328,  0.39739328,  0.39739328],
 [-0.33644763, -0.33644763, -0.33644763, -0.33644763]])

In [ ]:
new_array = np.apply_along_axis(relu, 0, l1)
new_array

Las formulas que necesitamos para implementar el backpropagation son:


\begin{align}
 \frac{\partial J}{\partial \mathbf{W_1}} &= \rm{ReLU}\left ( \mathbf{W_2^\top} (\mathbf{\hat{y}} - \mathbf{y})\right )\mathbf{x}^\top \\
 \frac{\partial J}{\partial \mathbf{W_2}} &= (\mathbf{\hat{y}} - \mathbf{y})\mathbf{h^\top} \\
 \frac{\partial J}{\partial \mathbf{b_1}} &= \rm{ReLU}\left ( \mathbf{W_2^\top} (\mathbf{\hat{y}} - \mathbf{y})\right ) \\
 \frac{\partial J}{\partial \mathbf{b_2}} &= \mathbf{\hat{y}} - \mathbf{y} 
\end{align}


In [ ]:
def back_prop(x, yhat, y, h, W1, W2, b1, b2, batch_size):

    # Calcular l1 as W2^T (Yhat - Y)
    #  W2^T (Yhat - Y) 
    l1 = np.dot(W2.T,(yhat-y))
    # Apply relu to l1
    l1 = np.apply_along_axis(relu, 0, l1)
    # Calcular el gradient of W1
    grad_W1 = (1/batch_size)*np.dot(l1,x.T) 
    # Calcular el gradient of W2
    grad_W2 = (1/batch_size)*np.dot(yhat-y,h.T)
    # Calcular el gradient of b1
    grad_b1 = np.sum((1/batch_size)*np.dot(l1,x.T),axis=1,keepdims=True)
    # Calcular el gradient of b2
    grad_b2 = np.sum((1/batch_size)*np.dot(yhat-y,h.T),axis=1,keepdims=True)
    
    return grad_W1, grad_W2, grad_b1, grad_b2

In [ ]:
# Probar la función
tmp_C = 2
tmp_N = 50
tmp_batch_size = 4
tmp_word2Ind, tmp_Ind2word = get_dict(data)
tmp_V = len(word2Ind)

# get a batch of data
tmp_x, tmp_y = next(get_batches(data, tmp_word2Ind, tmp_V,tmp_C, tmp_batch_size))

print("Tomando un batch de losd datos")
print(f"tmp_x.shape {tmp_x.shape}")
print(f"tmp_y.shape {tmp_y.shape}")

print()
print("Inicializando los pesos y los bias")
tmp_W1, tmp_W2, tmp_b1, tmp_b2 = initialize_model(tmp_N,tmp_V)

print(f"tmp_W1.shape {tmp_W1.shape}")
print(f"tmp_W2.shape {tmp_W2.shape}")
print(f"tmp_b1.shape {tmp_b1.shape}")
print(f"tmp_b2.shape {tmp_b2.shape}")

print()
print("Aplicando Forwad prop para obtener z y h")
tmp_z, tmp_h = forward_prop(tmp_x, tmp_W1, tmp_W2, tmp_b1, tmp_b2)
print(f"tmp_z.shape: {tmp_z.shape}")
print(f"tmp_h.shape: {tmp_h.shape}")

print()
print("Obtenemos yhat llamando la función de activación por softmax")
tmp_yhat = softmax(tmp_z)
print(f"tmp_yhat.shape: {tmp_yhat.shape}")

tmp_m = (2*tmp_C)
tmp_grad_W1, tmp_grad_W2, tmp_grad_b1, tmp_grad_b2 = back_prop(tmp_x, tmp_yhat, tmp_y, tmp_h, tmp_W1, tmp_W2, tmp_b1, tmp_b2, tmp_batch_size)

print()
print("call back_prop")
print(f"tmp_grad_W1.shape {tmp_grad_W1.shape}")
print(f"tmp_grad_W2.shape {tmp_grad_W2.shape}")
print(f"tmp_grad_b1.shape {tmp_grad_b1.shape}")
print(f"tmp_grad_b2.shape {tmp_grad_b2.shape}")


## Gradient Descent

Ahora que hemos implementado una función para calcular los gradientes, implementaremos el descenso de gradientes por **lotes** en su conjunto de entrenamiento.

Para esto, usaremos `initialize_model` y las funciones `back_prop` que acabamos de crear (y la función `compute_cost`). También podemos utilizar la función auxiliar `get_batches`:

```
for x, y in get_batches(data, word2Ind, V, C, batch_size):```

```...```


Durante la fase del gradiante descendente, se actualizarán los pesos y los bías $ \alpha $ veces el gradiente de las matrices y vectores originales, utilizando las siguientes fórmulas.


\begin{align}
 \mathbf{W_1} &:= \mathbf{W_1} - \alpha \frac{\partial J}{\partial \mathbf{W_1}} \\
 \mathbf{W_2} &:= \mathbf{W_2} - \alpha \frac{\partial J}{\partial \mathbf{W_2}} \\
 \mathbf{b_1} &:= \mathbf{b_1} - \alpha \frac{\partial J}{\partial \mathbf{b_1}}\\
 \mathbf{b_2} &:= \mathbf{b_2} - \alpha \frac{\partial J}{\partial \mathbf{b_2}} \\
\end{align}


In [ ]:
def gradient_descent(data, word2Ind, N, V, num_iters, alpha=0.03):

    W1, W2, b1, b2 = initialize_model(N,V, random_seed=282)
    batch_size = 128
    iters = 0
    C = 2
    for x, y in get_batches(data, word2Ind, V, C, batch_size):
        # Obtenemos z and h
        z, h = forward_prop(x, W1, W2, b1, b2)
        # Obteniendo yhat
        yhat = softmax(z)
        # calculando la función de costo
        cost = compute_cost(y, yhat, batch_size)
        if ( (iters+1) % 10 == 0):
            print(f"iters: {iters + 1} cost: {cost:.6f}")
        # Obteniendo los gradientes
        grad_W1, grad_W2, grad_b1, grad_b2 = back_prop(x, yhat, y, h, W1, W2, b1, b2, batch_size)
        
        # Actualizando los pesos
        W1 -= alpha*grad_W1 
        W2 -= alpha*grad_W2
        b1 -= alpha*grad_b1
        b2 -= alpha*grad_b2
        
        iters += 1 
        if iters == num_iters: 
            break
        if iters % 100 == 0:
            alpha *= 0.66
            
    return W1, W2, b1, b2

In [ ]:
# Probando la función
C = 2
N = 50
word2Ind, Ind2word = get_dict(tokens_data)
V = len(word2Ind)
num_iters = 150
print("Call gradient_descent")
W1, W2, b1, b2 = gradient_descent(tokens_data, word2Ind, N, V, num_iters)


## Visualización de los vectores palabra
En esta parte visualizaremos los vectores de palabras entrenados usando la función que acabas de codificar arriba.

#### Opción 1: extraer los embeddings de W1

In [ ]:
for i in range(V):
    print(Ind2word[i])

In [ ]:
V

In [ ]:
W1.T.shape

In [ ]:
words = ['lord','man', 'woman']
idx = [word2Ind[word] for word in words]
for i in idx:
    print(f'{Ind2word[i]} -----> {W1.T[i,:]}')

#### Opción 2: extraer los embeddings de W2

In [ ]:
W2.shape

In [ ]:
words = ['lord','man', 'woman']
idx = [word2Ind[word] for word in words]
for i in idx:
    print(f'{Ind2word[i]} -----> {W2[i,:]}')

#### Opción 3: extraer los embeddings de W1 y W2

In [ ]:
# visualizing the word vectors here
from matplotlib import pyplot
%config InlineBackend.figure_format = 'svg'
words = ['king', 'queen','lord','man', 'woman','dog','wolf',
         'rich','happy','sad']

#Vector de palabras (embeddings)
word_embs = (W1.T + W2)/2.0
 
# given a list of words and the embeddings, it returns a matrix with all the embeddings
idx = [word2Ind[word] for word in words]
X = word_embs[idx, :]
print(X.shape, idx)  # X.shape:  Number of words of dimension N each 

In [ ]:
result= compute_pca(X, 2)
pyplot.scatter(result[:, 0], result[:, 1])
for i, word in enumerate(words):
    pyplot.annotate(word, xy=(result[i, 0], result[i, 1]))
pyplot.show()

In [ ]:
result= compute_pca(X, 4)
pyplot.scatter(result[:, 3], result[:, 1])
for i, word in enumerate(words):
    pyplot.annotate(word, xy=(result[i, 3], result[i, 1]))
pyplot.show()